# 07 — AIA AlexNet Fold-2015 Results Comparison

This notebook compares the two completed AIA AlexNet fold-2015 benchmarks:

1. **05 Pilot benchmark** — smaller cost-controlled pilot.
2. **06B Largecap benchmark** — larger chronological fold-2015 benchmark using all positives and capped negatives.

The purpose is to create a paper-ready comparison table and interpretation before moving to stronger models such as ResNet18 or AIA+SHARP fusion.


In [ ]:
from pathlib import Path
import json
import pandas as pd
import subprocess

# Robust repo root detection. This works whether the notebook is executed
# from the repo root or from notebooks/training via nbconvert.
ROOT = Path(
    subprocess.check_output(["git", "rev-parse", "--show-toplevel"], text=True).strip()
)

PILOT = ROOT / "results/metrics/aia_alexnet_fold2015_benchmark_pilot_metrics.json"
LARGECAP = ROOT / "results/metrics/aia_alexnet_fold2015_largecap_benchmark_largecap_metrics.json"

print("Repo root:", ROOT)
print("Pilot exists:", PILOT.exists(), PILOT)
print("Largecap exists:", LARGECAP.exists(), LARGECAP)

assert PILOT.exists(), f"Missing pilot metrics: {PILOT}"
assert LARGECAP.exists(), f"Missing largecap metrics: {LARGECAP}"


In [ ]:
def load_json(path):
    with open(path) as f:
        return json.load(f)

pilot = load_json(PILOT)
largecap = load_json(LARGECAP)

def split_summary(metrics, split):
    rows = metrics["data_summary"]
    item = [r for r in rows if r["split"] == split][0]
    return item

def official_row(metrics, label):
    test = metrics["test"]["at_selected_threshold"]
    val = metrics["validation"]["best_tss"]
    train = split_summary(metrics, "train")
    val_split = split_summary(metrics, "val")
    test_split = split_summary(metrics, "test")
    
    return {
        "experiment_label": label,
        "experiment_name": metrics["experiment_name"],
        "run_mode": metrics["run_mode"],
        "fold_id": metrics["fold_id"],
        "train_rows": train["rows"],
        "train_positives": train["positives"],
        "train_positive_rate": train["positive_rate"],
        "val_rows": val_split["rows"],
        "val_positives": val_split["positives"],
        "val_positive_rate": val_split["positive_rate"],
        "test_rows": test_split["rows"],
        "test_positives": test_split["positives"],
        "test_positive_rate": test_split["positive_rate"],
        "best_epoch": metrics["best_epoch"],
        "validation_selected_threshold": metrics["selected_threshold_from_validation"],
        "val_roc_auc": metrics["validation"]["roc_auc"],
        "val_pr_auc": metrics["validation"]["pr_auc"],
        "val_tss": val["tss"],
        "val_hss": val["hss"],
        "test_roc_auc": metrics["test"]["roc_auc"],
        "test_pr_auc": metrics["test"]["pr_auc"],
        "test_brier": metrics["test"]["brier_score"],
        "test_accuracy": test["accuracy"],
        "test_precision": test["precision"],
        "test_recall": test["recall"],
        "test_specificity": test["specificity"],
        "test_f1": test["f1"],
        "test_tss": test["tss"],
        "test_hss": test["hss"],
        "tp": test["tp"],
        "tn": test["tn"],
        "fp": test["fp"],
        "fn": test["fn"],
    }

comparison = pd.DataFrame([
    official_row(pilot, "05 pilot"),
    official_row(largecap, "06B largecap"),
])

comparison


In [ ]:
display_cols = [
    "experiment_label", "run_mode", "train_rows", "val_rows", "test_rows",
    "test_positives", "validation_selected_threshold",
    "test_roc_auc", "test_pr_auc", "test_precision", "test_recall",
    "test_specificity", "test_f1", "test_tss", "test_hss",
    "tp", "tn", "fp", "fn"
]

comparison_display = comparison[display_cols].copy()

numeric_cols = comparison_display.select_dtypes(include="number").columns
comparison_display[numeric_cols] = comparison_display[numeric_cols].round(4)

comparison_display


In [ ]:
out_csv = ROOT / "results/metrics/aia_alexnet_fold2015_pilot_vs_largecap_comparison.csv"
out_md = ROOT / "results/metrics/aia_alexnet_fold2015_pilot_vs_largecap_comparison.md"

comparison_display.to_csv(out_csv, index=False)
out_md.write_text(comparison_display.to_markdown(index=False))

print("Saved CSV:", out_csv)
print("Saved Markdown:", out_md)


## Interpretation

The pilot benchmark demonstrated that the six-channel AIA AlexNet pipeline could learn a useful flare-risk signal under a smaller controlled setting.

The 06B largecap benchmark is more important scientifically because it uses the same chronological fold-2015 structure but expands the sample size substantially by keeping all positives and using capped negatives. This makes it a more realistic image-only baseline.

However, the larger benchmark also shows that AlexNet alone is not strong enough as the final model. The official validation-selected test result has useful recall, but false positives remain high and both TSS and HSS are limited. This supports the next modelling step: evaluate a stronger image encoder such as ResNet18 or EfficientNet-B0, then later compare against AIA+SHARP fusion.


In [ ]:
print("Notebook 07 complete.")
